### Import libraries

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime
from tsl.data import SpatioTemporalDataset

DATA_FOLDER = '../data/'

### Load data

In [ ]:

df = pd.read_csv(os.path.join(DATA_FOLDER, 'EWZ.csv'), sep=";")


# First row: AKS, I don't know what it is
AKS = df.iloc[0]
# Second row: Unit: MWh, constant value
df.drop(index=[0, 1], inplace=True)

# delta time: 15 minutes:
# 1 hour: 4 steps
# 1 day: 96 steps
# 1 week: 672 steps

print(df.shape)
df.head()

### Convert all values to float and clear zero-sensor

In [ ]:
EPS = 1e-12

# rename and cast to datetime
df.rename(columns={"Name":"Time"}, inplace=True)
df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, errors='coerce')
time = df['Time'].values

# Exclude name column
sensor_columns = df.columns[1:]
valid_column = []

print(f"before: {len(sensor_columns)} sensors")
for c in sensor_columns:
    df[c] = df[c].astype(np.float64)
    max_val = df[c].max()
    if max_val > EPS:
        valid_column.append(c)

sensor_columns = valid_column
df = df[['Time'] + valid_column]


print(f"after: {len(sensor_columns) } sensors")


### Glossary:

Wärmezähler: Heat meter

Schwimmbad: Pool

### Manual check of time series

In [ ]:
# result from visual inspection
sure = set([0, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52,
53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
66, 67, 68, 69, 70, 71, 78, 80, 81, 82, 83, 84, 85, 86,
87, 88, 89, 90, 91, 92, 93, 94, 95,
96])

keep = []

for idx, c in enumerate(sensor_columns):

    if idx in sure:
        keep.append(c)


    # Uncomment to plot daily means (too heavy to push to github)

    # step = 96  # plot every day
    # values = df[c].values
    
    # # daily means
    # time_tmp = []
    # values_tmp = []
    
    # for day_start in range(0, len(values), step):
    #     day_end = min(day_start + step, len(values))
        
    #     day_time = time[day_start]
        
    #     day_values = values[day_start:day_end]
    #     daily_mean = np.mean(day_values)
        
    #     time_tmp.append(day_time)
    #     values_tmp.append(daily_mean)

    # plt.plot(time_tmp, values_tmp)
    # plt.title(f"{idx}: {c}")
    # plt.xlabel("Time")
    # plt.ylabel("MWh (Daily Mean)")
    # plt.ylim(0, 1.1 * max(values_tmp))
    # plt.show()

# Filter 
df = df[['Time'] + keep]
sensor_columns = keep

print(f"final: {len(sensor_columns) } sensors")
    

In [ ]:


valid_col = []
dt = 9000
plot = True

print(f"before: {len(sensor_columns)} sensors")
for col in sensor_columns:


    # Filter zero values before computing mean and std

    values = df[col].values

    # do not take zero values into account for mean and std
    nonzero = values > EPS
    values_tmp = values[nonzero]
    time_tmp = time[nonzero]


    mean = np.mean(values_tmp)
    std = np.std(values_tmp)

    # Ignore outliers
    mask = (values <= mean + 3 * std) & (values >= 0) # do not take negative values
    values = values[mask]
    time_tmp = time[mask]

    max_after = np.max(values)

    # Replace outliers in the dataframe with the max_after value
    df.loc[~mask, col] = max_after


    max_power = np.argmax(values)
    max_date = df.iloc[max_power]['Time'].month
    
    # Check if max is in winter
    valid = max_date in [1, 2, 11, 12]
    if valid:
        valid_col.append(col)

    if plot and valid:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Time series plot
        values = df[col].values
        axes[0].plot(range(len(values)), values)
        axes[0].set_title(f"{col} - Time Domain")
        axes[0].set_xlabel("Time")
        axes[0].set_ylabel("Power [MWh]")



### Get intersection

In [ ]:
max_nz = datetime(2003, 1, 1) # date in the past
idx_intersection = -1

for col in sensor_columns:

    values = df[col].values
    nonzero = df[col] > EPS
    nz_idx = np.argmax(nonzero) # argmax takes first occurence of max
    first_nz = df.iloc[nz_idx]['Time']
    if first_nz > max_nz:
        max_nz = first_nz
        idx_intersection = nz_idx


max_nz


    

In [ ]:
df_intersection = df.iloc[idx_intersection:]


In [ ]:
df_intersection.set_index('Time', inplace=True)

df_intersection.head()

In [ ]:
# sanity check
torch_dataset = SpatioTemporalDataset(target=df_intersection,
                                      horizon=12,
                                      window=12,
                                      stride=1)


torch_dataset.shape


In [ ]:
df_intersection.to_csv(os.path.join(DATA_FOLDER, 'EWZ_cleaned.csv'))

In [ ]:


for col in df_intersection.columns:
    values = df_intersection[col].values
    plt.plot(range(len(values)), values)
    plt.title(col)
    plt.show()